# Module 5 — Code Along: REST API Client (the bank-accounts story)

We drive an accounts API from Python and validate every response into the M4 Pydantic model. There's **no live server** here — we **inject a `FakeSession`** that stands in for `requests.Session`. That dependency-injection seam is the same one the Day-3 tests mock and the Day-4 agent uses for its `llm_client`.

Section by section, one code cell per beat — run top to bottom:

**Section 1** HTTP with requests · **Section 2** decorators · **Section 3** the AccountClient

## Section 1 — HTTP with requests

A `Session` carries headers + a connection across calls; `timeout` guards against a hang; the status code tells you who broke. We use a `FakeSession` so it all runs offline.

In [ ]:
# 1.1 · The stand-in server (FakeSession) + a first request.
import json as _json

class FakeResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code; self._payload = payload
    @property
    def ok(self): return self.status_code < 400
    def json(self): return self._payload
    @property
    def text(self): return _json.dumps(self._payload)
    def raise_for_status(self):
        if not self.ok: raise RuntimeError(f"HTTP {self.status_code}")

class FakeSession:
    """Stands in for requests.Session — no network. Simulates the accounts API."""
    def __init__(self):
        self.headers = {}
        self._db = {1: {"id": 1, "owner": "Ada", "balance": 1500.0},
                    2: {"id": 2, "owner": "Lin", "balance": 800.0}}
    def request(self, method, url, json=None, timeout=None, **kw):
        tail = url.split("/accounts", 1)
        if len(tail) == 1:
            return FakeResponse(404, {"detail": "no route"})
        rest = tail[1].strip("/")
        if method == "GET" and rest == "":
            return FakeResponse(200, list(self._db.values()))
        if method == "GET":
            acc = self._db.get(int(rest))
            return FakeResponse(200, acc) if acc else FakeResponse(404, {"detail": "missing"})
        if method == "POST":
            self._db[json["id"]] = json
            return FakeResponse(201, json)
        return FakeResponse(405, {"detail": "method"})
    def get(self, url, **kw):  return self.request("GET", url, **kw)
    def post(self, url, **kw): return self.request("POST", url, **kw)

session = FakeSession()
r = session.get("http://server/accounts")
print(r.status_code, "->", r.json())     # 200 -> [Ada, Lin]

In [ ]:
# 1.2 · Status codes — who broke, and the retry rule.
print(session.get("http://server/accounts/1").status_code)    # 200 ok
print(session.get("http://server/accounts/99").status_code)   # 404 — YOUR fault: a retry gets 404 again
# rule: retry 5xx + network errors; never retry 4xx

In [ ]:
# 1.3 · A Session carries headers across calls; timeout guards a hang.
session.headers["X-Trace"] = "abc"                       # set once...
r = session.get("http://server/accounts", timeout=5)     # ...sent on every call; timeout always passed
print(r.ok, r.status_code, "| header set:", "X-Trace" in session.headers)

In [ ]:
# 1.4 · Auth — a token in a header, from the environment (never the URL, never source).
import os
os.environ.setdefault("TOKEN", "secret-123")
session.headers["Authorization"] = f"Bearer {os.environ['TOKEN']}"
print("Authorization" in session.headers)     # True — every later call now carries it

## Section 2 — Decorators: write your own

`@deco` above a function is just `f = deco(f)`. We write `@log_calls` (with `functools.wraps`) and `@retry` (a decorator that takes config), the resilience tool for §3.

In [ ]:
# 2.1 · @ is sugar: @deco above f is just  f = deco(f).
def shout(func):
    def wrapper(*a, **k): return str(func(*a, **k)).upper()
    return wrapper

@shout
def greet(n): return f"hi {n}"
# identical to:  greet = shout(greet)
print(greet("ada"))     # HI ADA

In [ ]:
# 2.2 · @log_calls — *args/**kwargs forward any call; functools.wraps keeps the real name.
import functools, logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("client")

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        log.info("call %s", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b): return a + b
print(add(2, 3), "| __name__:", add.__name__)   # logs 'call add'; 5 | add  (not 'wrapper')

In [ ]:
# 2.3 · @retry(times=3) — config layer; retries ONLY the listed exceptions.
import time
def retry(times=3, delay=0.0, exceptions=(Exception,)):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*a, **k):
            for attempt in range(1, times + 1):
                try:
                    return func(*a, **k)
                except exceptions:
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return deco

calls = {"n": 0}
@retry(times=3, exceptions=(ConnectionError,))
def flaky():
    calls["n"] += 1
    if calls["n"] < 3: raise ConnectionError("drop")
    return "ok"
print(flaky(), "after", calls["n"], "tries")     # ok after 3 tries

@retry(times=3, exceptions=(ConnectionError,))
def four04(): raise ValueError("400")
try:
    four04()
except ValueError:
    print("4xx not in the tuple -> not retried, raised at once")

## Section 3 — The AccountClient

One `_request` funnel owns url-join + timeout + error-check; `@retry` on it makes every call resilient; each method validates the JSON back into the M4 Pydantic model.

In [ ]:
# 3.1 · One _request funnel — url-join, timeout, error check -> APIError.
class APIError(Exception):
    def __init__(self, status, detail):
        super().__init__(f"{status}: {detail}"); self.status = status

class AccountClient:
    def __init__(self, base_url="http://server", session=None):
        self.base_url = base_url
        self._session = session or FakeSession()
    def _request(self, method, path, **kw):
        resp = self._session.request(method, self.base_url + path, timeout=5, **kw)
        if not resp.ok:
            raise APIError(resp.status_code, resp.text)
        return resp

client = AccountClient(session=FakeSession())
print(client._request("GET", "/accounts").status_code)    # 200
try:
    client._request("GET", "/accounts/99")
except APIError as e:
    print("APIError status:", e.status)                   # 404

In [ ]:
# 3.2 · @retry on _request -> every method inherits resilience (transient only).
class FlakyOnceSession(FakeSession):
    def __init__(self):
        super().__init__(); self._fail = True
    def request(self, method, url, **kw):
        if self._fail:                       # drop only the FIRST call
            self._fail = False
            raise ConnectionError("blip")
        return super().request(method, url, **kw)

class ResilientClient(AccountClient):
    @retry(times=3, delay=0.0, exceptions=(ConnectionError,))
    def _request(self, method, path, **kw):
        return super()._request(method, path, **kw)

c = ResilientClient(session=FlakyOnceSession())
print(c._request("GET", "/accounts").status_code)    # 200 — recovered after the blip

In [ ]:
# 3.3 · Typed returns — validate the JSON into a Pydantic model at the boundary.
from pydantic import BaseModel

class BankAccount(BaseModel):
    id: int
    owner: str
    balance: float

class TypedClient(AccountClient):
    def list_accounts(self) -> list[BankAccount]:
        data = self._request("GET", "/accounts").json()
        return [BankAccount.model_validate(r) for r in data]      # validated here
    def get_account(self, id) -> BankAccount:
        return BankAccount.model_validate(self._request("GET", f"/accounts/{id}").json())

api = TypedClient(session=FakeSession())
accts = api.list_accounts()
print(type(accts[0]).__name__, "|", accts[0].owner, accts[0].balance)   # BankAccount | Ada 1500.0
print("get one:", api.get_account(2).owner)                              # Lin

## Next: Module 6

We now have an `AccountClient` that calls the API, retries transient failures, and returns **validated** Pydantic objects.

**Next we put it to work on data:** read a CSV of accounts, validate each row, `POST` it through the client, and hand back a **report** of what was created vs. what failed — the data-driven bulk-import workflow (and Day-3's system-under-test).